In [ ]:
# ============================================================
# NewsGroupsTopicEval
# Notebook NG03
#
# Representation Learning
#
# This notebook:
#
# 1. Loads cross-validation folds
# 2. Restores token lists
# 3. Creates fold-specific Gensim dictionaries
# 4. Creates Bag-of-Words corpora
# 5. Trains LDA models for K = 2 ... 100
# 6. Extracts document-topic representations
# 7. Saves models and representations
#
# IMPORTANT:
# ----------
# Dictionary creation is performed independently for each fold
# using TRAINING DATA ONLY.
#
# This prevents information leakage from test documents.
#
# Classification is NOT performed in this notebook.
#
# Author: Mahedi Hasan
# ============================================================

In [ ]:
!pip -q install \
gensim \
pyLDAvis \
openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 64.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# Imports
# ============================================================

import os
import gc
import json
import time
import pickle
import random
import logging
import warnings
import sys
import ast

from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

from tqdm.auto import tqdm

import gensim

from gensim import corpora

from gensim.models import LdaModel
from gensim.models import CoherenceModel

from datetime import datetime


print("=" * 60)

print("Python :", sys.version)
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("Sklearn:", sklearn.__version__)
print("Gensim :", gensim.__version__)

print("=" * 60)


warnings.filterwarnings("ignore")

Python : 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
NumPy  : 2.1.3
Pandas : 2.2.3
Sklearn: 1.6.1
Gensim : 4.4.0


In [ ]:
# ============================================================
# Load Configuration
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval"
)


CONFIG_FILE = (
    PROJECT_ROOT
    / "config"
    / "config.json"
)


with open(CONFIG_FILE) as f:

    CONFIG = json.load(f)


print("Configuration Loaded")

Configuration Loaded


In [ ]:
# ============================================================
# NG03-6 — Load and Validate Experiment Configuration
# ============================================================


print(f"Project       : {CONFIG.get('project_name')}")
print(f"Dataset       : {CONFIG.get('dataset_name')}")
print(f"Dataset API   : {CONFIG.get('dataset_api')}")
print(f"Random seed   : {CONFIG.get('random_seed')}")
print(f"Number folds  : {CONFIG.get('n_folds')}")
print()

# ------------------------------------------------------------
# 3. Dataset configuration
# ------------------------------------------------------------

TEXT_COLUMN = CONFIG["text_column"]
LABEL_COLUMN = CONFIG["label_column"]
LABEL_ID_COLUMN = CONFIG["label_id_column"]
DOCUMENT_ID_COLUMN = CONFIG["document_id_column"]

RANDOM_SEED = CONFIG["random_seed"]
N_FOLDS = CONFIG["n_folds"]

# ------------------------------------------------------------
# 4. Preprocessing configuration
# ------------------------------------------------------------

REMOVE_BELOW = CONFIG["remove_below"]
REMOVE_ABOVE = CONFIG["remove_above"]

# ------------------------------------------------------------
# 5. Topic range
# ------------------------------------------------------------

TOPIC_MIN = CONFIG["topic_min"]
TOPIC_MAX = CONFIG["topic_max"]

TOPIC_LIST = list(range(TOPIC_MIN, TOPIC_MAX + 1))

# ------------------------------------------------------------
# 6. LDA configuration
# ------------------------------------------------------------
# IMPORTANT:
# The NG config.json stores these as top-level keys:
#
# lda_passes
# lda_iterations
# lda_chunksize
# lda_alpha
# lda_eta
# lda_update_every
# lda_decay
# lda_offset
# lda_minimum_probability
# lda_per_word_topics
# lda_eval_every
# lda_dtype
# ------------------------------------------------------------

LDA_PASSES = CONFIG["lda_passes"]
LDA_ITERATIONS = CONFIG["lda_iterations"]
LDA_CHUNKSIZE = CONFIG["lda_chunksize"]

ALPHA = CONFIG["lda_alpha"]
ETA = CONFIG["lda_eta"]

LDA_UPDATE_EVERY = CONFIG["lda_update_every"]
LDA_DECAY = CONFIG["lda_decay"]
LDA_OFFSET = CONFIG["lda_offset"]
LDA_MINIMUM_PROBABILITY = CONFIG["lda_minimum_probability"]
LDA_PER_WORD_TOPICS = CONFIG["lda_per_word_topics"]
LDA_EVAL_EVERY = CONFIG["lda_eval_every"]
LDA_DTYPE = CONFIG["lda_dtype"]

TOP_WORDS = CONFIG["top_words"]

# ------------------------------------------------------------
# 7. Display configuration
# ------------------------------------------------------------

print("=" * 70)
print("DATASET CONFIGURATION")
print("=" * 70)

print(f"Text column          : {TEXT_COLUMN}")
print(f"Label column         : {LABEL_COLUMN}")
print(f"Label ID column      : {LABEL_ID_COLUMN}")
print(f"Document ID column   : {DOCUMENT_ID_COLUMN}")
print(f"Random seed          : {RANDOM_SEED}")
print(f"Number of folds      : {N_FOLDS}")
print()

print("=" * 70)
print("PREPROCESSING CONFIGURATION")
print("=" * 70)

print(f"Remove below         : {REMOVE_BELOW}")
print(f"Remove above         : {REMOVE_ABOVE}")
print()

print("=" * 70)
print("TOPIC CONFIGURATION")
print("=" * 70)

print(f"Topic minimum        : {TOPIC_MIN}")
print(f"Topic maximum        : {TOPIC_MAX}")
print(f"Number of K values   : {len(TOPIC_LIST)}")
print(f"K range              : {TOPIC_LIST[0]} ... {TOPIC_LIST[-1]}")
print()

print("=" * 70)
print("LDA CONFIGURATION")
print("=" * 70)

print(f"Passes               : {LDA_PASSES}")
print(f"Iterations           : {LDA_ITERATIONS}")
print(f"Chunksize            : {LDA_CHUNKSIZE}")
print(f"Alpha                : {ALPHA}")
print(f"Eta                  : {ETA}")
print(f"Update every         : {LDA_UPDATE_EVERY}")
print(f"Decay                : {LDA_DECAY}")
print(f"Offset               : {LDA_OFFSET}")
print(f"Minimum probability  : {LDA_MINIMUM_PROBABILITY}")
print(f"Per-word topics      : {LDA_PER_WORD_TOPICS}")
print(f"Eval every           : {LDA_EVAL_EVERY}")
print(f"Dtype                : {LDA_DTYPE}")
print()

print("=" * 70)
print("CONFIGURATION VALIDATION")
print("=" * 70)

# Required checks
assert isinstance(TOPIC_MIN, int)
assert isinstance(TOPIC_MAX, int)
assert TOPIC_MIN <= TOPIC_MAX

assert N_FOLDS >= 2

assert LDA_PASSES > 0
assert LDA_ITERATIONS > 0
assert LDA_CHUNKSIZE > 0

# Critical reproducibility check
assert ALPHA == "symmetric", (
    f"Expected lda_alpha='symmetric', but found {ALPHA!r}"
)

assert ETA == "symmetric", (
    f"Expected lda_eta='symmetric', but found {ETA!r}"
)

print("✓ Configuration file exists")
print("✓ Dataset configuration loaded")
print("✓ Topic range validated")
print("✓ LDA configuration validated")
print("✓ Alpha = symmetric")
print("✓ Eta   = symmetric")
print()
print("CONFIGURATION IS READY FOR EXPERIMENTS.")

Project       : NewsGroupsTopicEval
Dataset       : 20_Newsgroups
Dataset API   : fetch_20newsgroups
Random seed   : 42
Number folds  : 5

DATASET CONFIGURATION
Text column          : Text
Label column         : Label
Label ID column      : LabelID
Document ID column   : DocumentID
Random seed          : 42
Number of folds      : 5

PREPROCESSING CONFIGURATION
Remove below         : 5
Remove above         : 0.5

TOPIC CONFIGURATION
Topic minimum        : 2
Topic maximum        : 100
Number of K values   : 99
K range              : 2 ... 100

LDA CONFIGURATION
Passes               : 20
Iterations           : 400
Chunksize            : 2000
Alpha                : symmetric
Eta                  : symmetric
Update every         : 1
Decay                : 0.5
Offset               : 1.0
Minimum probability  : 0.0
Per-word topics      : False
Eval every           : None
Dtype                : float32

CONFIGURATION VALIDATION
✓ Configuration file exists
✓ Dataset configuration loaded
✓ Topic 

In [ ]:
# ============================================================
# Topic Range
# ============================================================

TOPIC_LIST = list(

    range(

        TOPIC_MIN,

        TOPIC_MAX + 1

    )
)


print(
    "Topic Range:",
    TOPIC_MIN,
    "to",
    TOPIC_MAX
)


print(
    "Total Topic Numbers =",
    len(TOPIC_LIST)
)

Topic Range: 2 to 100
Total Topic Numbers = 99


In [ ]:
# ============================================================
# Output Directories
# ============================================================

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)


VECTOR_DIR = (
    PROJECT_ROOT
    / "topic_vectors"
)


METRIC_DIR = (
    PROJECT_ROOT
    / "intrinsic_metrics"
)


LOG_DIR = (
    PROJECT_ROOT
    / "logs"
)


DICTIONARY_DIR = (
    PROJECT_ROOT
    / "dictionary"
)


for directory in [

    MODEL_DIR,

    VECTOR_DIR,

    METRIC_DIR,

    LOG_DIR,

    DICTIONARY_DIR

]:

    directory.mkdir(

        parents=True,

        exist_ok=True
    )


print("Directories Ready")

Directories Ready


In [ ]:
# ============================================================
# Logger
# ============================================================

LOG_FILE = (

    LOG_DIR
    / "Notebook_NG03.log"
)


logging.basicConfig(

    filename=LOG_FILE,

    level=logging.INFO,

    format="%(asctime)s %(message)s"

)


logging.info(
    "Notebook NG03 Started"
)


print(LOG_FILE)

/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/logs/Notebook_NG03.log


In [ ]:
# ============================================================
# Reproducibility
# ============================================================

random.seed(
    RANDOM_SEED
)


np.random.seed(
    RANDOM_SEED
)


os.environ[
    "PYTHONHASHSEED"
] = str(
    RANDOM_SEED
)


print(
    "Seed =",
    RANDOM_SEED
)

Seed = 42


In [ ]:
def print_line():

    print(
        "=" * 70
    )

In [ ]:
# ============================================================
# Restore Tokens from Excel
# ============================================================

def restore_tokens(value):


    # Already a list
    if isinstance(value, list):

        return value


    # Missing value
    if pd.isna(value):

        return []


    # String representation of Python list
    if isinstance(value, str):

        try:

            tokens = ast.literal_eval(
                value
            )


            if isinstance(
                tokens,
                list
            ):

                return [

                    str(token)

                    for token in tokens
                ]


        except:

            pass


        # Fallback
        return value.split()


    return []

In [ ]:
# ============================================================
# Load Fold
# ============================================================

def load_fold(fold):


    train_file = (

        PROJECT_ROOT
        / "folds"
        / f"Fold_{fold}"
        / "train.xlsx"
    )


    test_file = (

        PROJECT_ROOT
        / "folds"
        / f"Fold_{fold}"
        / "test.xlsx"
    )


    train = pd.read_excel(
        train_file
    )


    test = pd.read_excel(
        test_file
    )


    # --------------------------------------------------------
    # Restore token lists
    # --------------------------------------------------------

    train["Tokens"] = (

        train["Tokens"]
        .apply(
            restore_tokens
        )
    )


    test["Tokens"] = (

        test["Tokens"]
        .apply(
            restore_tokens
        )
    )


    return train, test

In [ ]:
# ============================================================
# Validate Fold Loading
# ============================================================

train, test = load_fold(1)


print(
    "Train Shape:",
    train.shape
)


print(
    "Test Shape:",
    test.shape
)


print()

print(
    "Train Token Example:"
)


print(
    train["Tokens"].iloc[0][:20]
)


print()

print(
    "Test Token Example:"
)


print(
    test["Tokens"].iloc[0][:20]
)


assert isinstance(
    train["Tokens"].iloc[0],
    list
)


assert isinstance(
    test["Tokens"].iloc[0],
    list
)


print()

print(
    "✓ Token restoration successful"
)

Train Shape: (14637, 7)
Test Shape: (3660, 7)

Train Token Example:
['sure', 'bashers', 'pen', 'fan', 'pretty', 'confused', 'lack', 'kind', 'post', 'recent', 'pen', 'massacre', 'devil', 'actually', 'bit', 'puzzled', 'bit', 'relieved', 'however', 'going']

Test Token Example:
['brother', 'market', 'high', 'performance', 'video', 'card', 'support', 'vesa', 'local', 'bus', 'mb', 'ram', 'anyone', 'suggestion', 'idea', 'diamond', 'stealth', 'pro', 'local', 'bus']

✓ Token restoration successful


In [ ]:
# ============================================================
# Build Fold-Specific Dictionary
# ============================================================

def build_dictionary(

    train_tokens

):


    dictionary = corpora.Dictionary(

        train_tokens
    )


    dictionary.filter_extremes(

        no_below=REMOVE_BELOW,

        no_above=REMOVE_ABOVE
    )


    dictionary.compactify()


    return dictionary

In [ ]:
# ============================================================
# Dictionary Validation
# ============================================================

train, test = load_fold(1)


train_tokens = (

    train["Tokens"]
    .tolist()
)


dictionary = build_dictionary(

    train_tokens
)


print(
    "Dictionary Size:",
    len(dictionary)
)


assert len(dictionary) > 0


print(
    "✓ Fold-specific dictionary successfully created"
)

Dictionary Size: 18264
✓ Fold-specific dictionary successfully created


In [ ]:
# ============================================================
# Create Corpus
# ============================================================

def create_corpus(

    tokens,

    dictionary

):


    corpus = [

        dictionary.doc2bow(
            document
        )

        for document in tokens
    ]


    return corpus

In [ ]:
# ============================================================
# Corpus Validation
# ============================================================

train_tokens = (

    train["Tokens"]
    .tolist()
)


test_tokens = (

    test["Tokens"]
    .tolist()
)


train_corpus = create_corpus(

    train_tokens,

    dictionary
)


test_corpus = create_corpus(

    test_tokens,

    dictionary
)


print(
    "Train Documents:",
    len(train_corpus)
)


print(
    "Test Documents:",
    len(test_corpus)
)


print()

print(
    "First Train BOW:"
)


print(
    train_corpus[0][:10]
)


print()

print(
    "✓ Corpus creation successful"
)

Train Documents: 14637
Test Documents: 3660

First Train BOW:
[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 3), (7, 1), (8, 1), (9, 1)]

✓ Corpus creation successful


In [ ]:
# ============================================================
# Save Dictionary
# ============================================================

def save_dictionary(

    dictionary,

    fold

):


    fold_dictionary_dir = (

        DICTIONARY_DIR
        / f"Fold_{fold}"
    )


    fold_dictionary_dir.mkdir(

        parents=True,

        exist_ok=True
    )


    dictionary_file = (

        fold_dictionary_dir
        / "dictionary.dict"
    )


    dictionary.save(

        str(dictionary_file)
    )


    return dictionary_file

In [ ]:
# ============================================================
# Train LDA Model
# ============================================================

def train_lda_model(

    corpus,

    dictionary,

    num_topics

):


    model = LdaModel(

        corpus=corpus,

        id2word=dictionary,

        num_topics=num_topics,

        random_state=RANDOM_SEED,

        chunksize=LDA_CHUNKSIZE,

        passes=LDA_PASSES,

        iterations=LDA_ITERATIONS,

        alpha=ALPHA,

        eta=ETA,

        update_every=LDA_UPDATE_EVERY,

        decay=LDA_DECAY,

        offset=LDA_OFFSET,

        minimum_probability=LDA_MINIMUM_PROBABILITY,

        per_word_topics=LDA_PER_WORD_TOPICS,

        eval_every=LDA_EVAL_EVERY,

        dtype=LDA_DTYPE

    )


    return model

In [ ]:
# ============================================================
# Train Single LDA Model as Test
# ============================================================

test_topics = 10


print(
    "Training Single LDA Model..."
)


test_model = train_lda_model(

    train_corpus,

    dictionary,

    test_topics
)


print()

print(
    "✓ Test LDA Model Successfully Trained"
)


print()

print(
    test_model.print_topics(
        num_topics=5,
        num_words=10
    )
)

Training Single LDA Model...

✓ Test LDA Model Successfully Trained

[(np.int64(1), '0.264*"\'ax\'," + 0.033*"\'g\'," + 0.029*"\'v\'," + 0.028*"\'f\'," + 0.026*"\'r\'," + 0.022*"\'b\'," + 0.022*"\'u\'," + 0.022*"\'p\'," + 0.021*"\'c\'," + 0.018*"\'w\',"'), (np.int64(5), '0.033*"\'x\'," + 0.021*"\'q\'," + 0.013*"\'max\'," + 0.012*"\'image\'," + 0.011*"\'file\'," + 0.009*"\'do\'," + 0.008*"\'use\'," + 0.008*"\'program\'," + 0.007*"\'system\'," + 0.006*"\'window\',"'), (np.int64(7), '0.007*"psalm" + 0.006*"right" + 0.006*"water" + 0.004*"second" + 0.004*"power" + 0.004*"boy" + 0.004*"daughter" + 0.003*"pen" + 0.003*"ra" + 0.003*"arm"'), (np.int64(9), '0.012*"university" + 0.012*"new" + 0.010*"book" + 0.008*"research" + 0.007*"study" + 0.007*"st" + 0.006*"dr" + 0.006*"patient" + 0.006*"disease" + 0.006*"center"'), (np.int64(6), '0.020*"game" + 0.013*"team" + 0.011*"year" + 0.008*"player" + 0.006*"play" + 0.006*"first" + 0.006*"last" + 0.006*"win" + 0.005*"think" + 0.005*"get"')]


In [ ]:
# ============================================================
# NG03-22 — Initialize Experiment Manifest
# ============================================================

MANIFEST_DIR = PROJECT_ROOT / "models"

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


EXPERIMENT_MANIFEST_FILE = (
    MANIFEST_DIR
    / "experiment_manifest_stage3.csv"
)


manifest_columns = [

    "ExperimentID",
    "Fold",
    "Topics",
    "TrainDocuments",
    "TestDocuments",
    "DictionarySize",
    "ModelPath",
    "TrainVectorPath",
    "TestVectorPath",
    "Status"
]


experiment_manifest = pd.DataFrame(
    columns=manifest_columns
)


print(
    "Experiment manifest initialized"
)

print(
    "Expected experiments:",
    N_FOLDS * len(TOPIC_LIST)
)

Experiment manifest initialized
Expected experiments: 495


In [ ]:
# ============================================================
# NG03-23 — Dense Topic Vector Extraction
# ============================================================

def extract_topic_vectors(
    model,
    corpus,
    num_topics
):

    vectors = []

    for document in corpus:

        topic_distribution = model.get_document_topics(
            document,
            minimum_probability=0
        )

        vector = np.zeros(
            num_topics,
            dtype=np.float32
        )

        for topic_id, probability in topic_distribution:

            vector[
                topic_id
            ] = probability


        vectors.append(
            vector
        )


    return np.array(
        vectors,
        dtype=np.float32
    )

In [ ]:
# ============================================================
# NG03-24 — Validate Topic Vectors
# ============================================================

sample_vectors = extract_topic_vectors(

    model=test_model,

    corpus=test_corpus[:5],

    num_topics=test_topics
)


print(
    "Topic Vector Shape:",
    sample_vectors.shape
)


print()

print(
    "First Vector:"
)


print(
    sample_vectors[0]
)


print()

print(
    "Probability Sum:"
)


print(
    sample_vectors[0].sum()
)


assert sample_vectors.shape == (

    5,

    test_topics

)


print()

print(
    "✓ Topic vector extraction successful"
)

NameError: name 'test_model' is not defined

In [ ]:
# ============================================================
# NG03-25 — Save Topic Vectors
# ============================================================

def save_topic_vectors(

    vectors,

    fold,

    topics,

    dataset_type

):

    output_directory = (

        VECTOR_DIR
        / f"Fold_{fold}"
        / f"K_{topics}"
    )


    output_directory.mkdir(

        parents=True,

        exist_ok=True
    )


    vector_file = (

        output_directory
        / f"{dataset_type}_topic_vectors.npy"
    )


    np.save(

        vector_file,

        vectors
    )


    return vector_file

In [ ]:
# ============================================================
# NG03-26 — Save LDA Model
# ============================================================

def save_lda_model(

    model,

    fold,

    topics

):

    output_directory = (

        MODEL_DIR
        / f"Fold_{fold}"
        / f"K_{topics}"
    )


    output_directory.mkdir(

        parents=True,

        exist_ok=True
    )


    model_file = (

        output_directory
        / "lda_model"
    )


    model.save(
        str(model_file)
    )


    return model_file

In [ ]:
# ============================================================
# NG03-27 — Run Single LDA Experiment
# ============================================================

def run_single_experiment(

    fold,

    topics,

    train_corpus,

    test_corpus,

    dictionary

):


    # --------------------------------------------------------
    # Train LDA Model
    # --------------------------------------------------------

    model = train_lda_model(

        corpus=train_corpus,

        dictionary=dictionary,

        num_topics=topics
    )


    # --------------------------------------------------------
    # Extract Train Topic Vectors
    # --------------------------------------------------------

    train_vectors = extract_topic_vectors(

        model=model,

        corpus=train_corpus,

        num_topics=topics
    )


    # --------------------------------------------------------
    # Extract Test Topic Vectors
    # --------------------------------------------------------

    test_vectors = extract_topic_vectors(

        model=model,

        corpus=test_corpus,

        num_topics=topics
    )


    # --------------------------------------------------------
    # Save Model
    # --------------------------------------------------------

    model_file = save_lda_model(

        model=model,

        fold=fold,

        topics=topics
    )


    # --------------------------------------------------------
    # Save Train Vectors
    # --------------------------------------------------------

    train_vector_file = save_topic_vectors(

        vectors=train_vectors,

        fold=fold,

        topics=topics,

        dataset_type="train"
    )


    # --------------------------------------------------------
    # Save Test Vectors
    # --------------------------------------------------------

    test_vector_file = save_topic_vectors(

        vectors=test_vectors,

        fold=fold,

        topics=topics,

        dataset_type="test"
    )


    return {

        "Model": model,

        "ModelPath": str(
            model_file
        ),

        "TrainVectorPath": str(
            train_vector_file
        ),

        "TestVectorPath": str(
            test_vector_file
        )

    }

In [ ]:
# ============================================================
# NG03-28 — Validation Experiment
# ============================================================

VALIDATION_FOLD = 1

VALIDATION_K = 10


print("=" * 70)

print(
    f"VALIDATION EXPERIMENT"
)

print(
    f"Fold = {VALIDATION_FOLD}"
)

print(
    f"K = {VALIDATION_K}"
)

print("=" * 70)


# Load fold

train_df, test_df = load_fold(
    VALIDATION_FOLD
)


train_tokens = (
    train_df["Tokens"].tolist()
)


test_tokens = (
    test_df["Tokens"].tolist()
)


# Create dictionary

dictionary = build_dictionary(
    train_tokens
)


# Create corpora

train_corpus = create_corpus(

    train_tokens,

    dictionary
)


test_corpus = create_corpus(

    test_tokens,

    dictionary
)


# Save dictionary

dictionary_file = save_dictionary(

    dictionary,

    VALIDATION_FOLD
)


# Run experiment

validation_result = run_single_experiment(

    fold=VALIDATION_FOLD,

    topics=VALIDATION_K,

    train_corpus=train_corpus,

    test_corpus=test_corpus,

    dictionary=dictionary
)


print()

print("=" * 70)

print(
    "✓ VALIDATION EXPERIMENT COMPLETED"
)

print("=" * 70)


print()

print(
    "Dictionary:",
    dictionary_file
)


print(
    "Model:",
    validation_result["ModelPath"]
)


print(
    "Train vectors:",
    validation_result["TrainVectorPath"]
)


print(
    "Test vectors:",
    validation_result["TestVectorPath"]
)

VALIDATION EXPERIMENT
Fold = 1
K = 10


KeyboardInterrupt: 

In [ ]:

# ============================================================
# NG03-29 — Prepare Resources for All Folds
# ============================================================

print("=" * 70)
print("PREPARING ALL FOLD RESOURCES")
print("=" * 70)

fold_resources = {}

for fold in range(1, N_FOLDS + 1):

    print(f"\nPreparing Fold {fold}...")

    # --------------------------------------------------------
    # Load fold data
    # --------------------------------------------------------

    train_df, test_df = load_fold(fold)

    train_tokens = train_df["Tokens"].tolist()

    test_tokens = test_df["Tokens"].tolist()


    # --------------------------------------------------------
    # Build dictionary
    # --------------------------------------------------------

    dictionary = build_dictionary(
        train_tokens
    )


    # --------------------------------------------------------
    # Create BOW corpora
    # --------------------------------------------------------

    train_corpus = create_corpus(
        train_tokens,
        dictionary
    )

    test_corpus = create_corpus(
        test_tokens,
        dictionary
    )


    # --------------------------------------------------------
    # Save dictionary
    # --------------------------------------------------------

    dictionary_file = save_dictionary(
        dictionary,
        fold
    )


    # --------------------------------------------------------
    # Store resources
    # --------------------------------------------------------

    fold_resources[fold] = {

        "TrainCorpus": train_corpus,

        "TestCorpus": test_corpus,

        "Dictionary": dictionary,

        "TrainDocuments": len(train_df),

        "TestDocuments": len(test_df),

        "DictionarySize": len(dictionary),

        "DictionaryPath": str(dictionary_file)
    }


    print(
        f"✓ Fold {fold} prepared | "
        f"Train={len(train_df)} | "
        f"Test={len(test_df)} | "
        f"Dictionary={len(dictionary)}"
    )


print("\n" + "=" * 70)

print("✓ ALL FOLD RESOURCES PREPARED SUCCESSFULLY")

print("=" * 70)

PREPARING ALL FOLD RESOURCES

Preparing Fold 1...
✓ Fold 1 prepared | Train=14637 | Test=3660 | Dictionary=18264

Preparing Fold 2...
✓ Fold 2 prepared | Train=14637 | Test=3660 | Dictionary=17976

Preparing Fold 3...
✓ Fold 3 prepared | Train=14638 | Test=3659 | Dictionary=17589

Preparing Fold 4...
✓ Fold 4 prepared | Train=14638 | Test=3659 | Dictionary=17883

Preparing Fold 5...
✓ Fold 5 prepared | Train=14638 | Test=3659 | Dictionary=18083

✓ ALL FOLD RESOURCES PREPARED SUCCESSFULLY


In [ ]:

# ============================================================
# NG03-30 — Experiment Checkpoint System
# ============================================================

CHECKPOINT_FILE = (
    MODEL_DIR
    / "ng03_experiment_checkpoint.csv"
)


def load_completed_experiments():

    if CHECKPOINT_FILE.exists():

        checkpoint_df = pd.read_csv(
            CHECKPOINT_FILE
        )

        print(
            f"✓ Existing checkpoint loaded: "
            f"{len(checkpoint_df)} experiments"
        )

    else:

        checkpoint_df = pd.DataFrame(
            columns=[
                "ExperimentID",
                "Fold",
                "Topics",
                "TrainDocuments",
                "TestDocuments",
                "DictionarySize",
                "ModelPath",
                "TrainVectorPath",
                "TestVectorPath",
                "Status"
            ]
        )

        print(
            "✓ New checkpoint initialized"
        )


    return checkpoint_df


checkpoint_df = load_completed_experiments()


completed_experiments = set(

    zip(

        checkpoint_df["Fold"],

        checkpoint_df["Topics"]

    )

)


print()

print(
    "Completed experiments:",
    len(completed_experiments)
)


print(
    "Expected experiments:",
    N_FOLDS * len(TOPIC_LIST)
)

✓ Existing checkpoint loaded: 474 experiments

Completed experiments: 474
Expected experiments: 495


In [ ]:
# ============================================================
# RESET NG03 EXPERIMENT OUTPUTS
# ============================================================

import shutil
from pathlib import Path

print("=" * 70)
print("RESETTING NG03 EXPERIMENT OUTPUTS")
print("=" * 70)

# Remove checkpoint
if CHECKPOINT_FILE.exists():
    CHECKPOINT_FILE.unlink()
    print("✓ Deleted checkpoint:", CHECKPOINT_FILE)

# Remove generated LDA models
if MODEL_DIR.exists():
    for fold_dir in MODEL_DIR.glob("Fold_*"):
        shutil.rmtree(fold_dir)
        print("✓ Deleted:", fold_dir)

# Remove generated topic vectors
if VECTOR_DIR.exists():
    for fold_dir in VECTOR_DIR.glob("Fold_*"):
        shutil.rmtree(fold_dir)
        print("✓ Deleted:", fold_dir)

print()
print("=" * 70)
print("✓ NG03 EXPERIMENT OUTPUT RESET COMPLETE")
print("=" * 70)

RESETTING NG03 EXPERIMENT OUTPUTS
✓ Deleted checkpoint: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/models/ng03_experiment_checkpoint.csv
✓ Deleted: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/models/Fold_1
✓ Deleted: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/topic_vectors/Fold_1

✓ NG03 EXPERIMENT OUTPUT RESET COMPLETE


In [ ]:

# ============================================================
# NG03-31 — Full LDA Experiment Pipeline
# ============================================================

print("=" * 70)
print("STARTING FULL LDA EXPERIMENT PIPELINE")
print("=" * 70)


total_experiments = (

    N_FOLDS
    * len(TOPIC_LIST)

)


completed_count = len(
    completed_experiments
)


print(
    f"Total experiments: {total_experiments}"
)


print(
    f"Already completed: {completed_count}"
)


print(
    f"Remaining: "
    f"{total_experiments - completed_count}"
)


print()


# ============================================================
# EXPERIMENT LOOP
# ============================================================

for fold in range(1, N_FOLDS + 1):


    print("\n" + "=" * 70)

    print(
        f"PROCESSING FOLD {fold}"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # Load prepared resources
    # --------------------------------------------------------

    resources = fold_resources[
        fold
    ]


    train_corpus = resources[
        "TrainCorpus"
    ]


    test_corpus = resources[
        "TestCorpus"
    ]


    dictionary = resources[
        "Dictionary"
    ]


    # --------------------------------------------------------
    # Run every K
    # --------------------------------------------------------

    for topics in TOPIC_LIST:


        # ----------------------------------------------------
        # Skip completed experiment
        # ----------------------------------------------------

        if (

            fold,
            topics

        ) in completed_experiments:


            print(

                f"Fold {fold}, "
                f"K={topics}: "

                "Already completed — skipping"

            )


            continue


        print(

            f"\nRunning "

            f"Fold={fold}, "

            f"K={topics}"

        )


        try:


            # ------------------------------------------------
            # Run experiment
            # ------------------------------------------------

            result = run_single_experiment(

                fold=fold,

                topics=topics,

                train_corpus=train_corpus,

                test_corpus=test_corpus,

                dictionary=dictionary

            )


            # ------------------------------------------------
            # Create Experiment ID
            # ------------------------------------------------

            experiment_id = (

                f"NG_F{fold}_K{topics}"

            )


            # ------------------------------------------------
            # Create result row
            # ------------------------------------------------

            experiment_row = {

                "ExperimentID": experiment_id,

                "Fold": fold,

                "Topics": topics,

                "TrainDocuments":
                    resources[
                        "TrainDocuments"
                    ],

                "TestDocuments":
                    resources[
                        "TestDocuments"
                    ],

                "DictionarySize":
                    resources[
                        "DictionarySize"
                    ],

                "ModelPath":
                    result[
                        "ModelPath"
                    ],

                "TrainVectorPath":
                    result[
                        "TrainVectorPath"
                    ],

                "TestVectorPath":
                    result[
                        "TestVectorPath"
                    ],

                "Status":
                    "Completed"
            }


            # ------------------------------------------------
            # Add to checkpoint
            # ------------------------------------------------

            checkpoint_df = pd.concat(

                [

                    checkpoint_df,

                    pd.DataFrame(
                        [experiment_row]
                    )

                ],

                ignore_index=True

            )


            # ------------------------------------------------
            # Save checkpoint immediately
            # ------------------------------------------------

            checkpoint_df.to_csv(

                CHECKPOINT_FILE,

                index=False

            )


            # ------------------------------------------------
            # Update completed experiments
            # ------------------------------------------------

            completed_experiments.add(

                (

                    fold,

                    topics

                )

            )


            print(

                f"✓ Completed "

                f"Fold={fold}, "

                f"K={topics}"

            )


        except Exception as e:


            print(

                f"✗ ERROR "

                f"Fold={fold}, "

                f"K={topics}"

            )


            print(

                str(e)

            )


            # Save checkpoint before continuing

            checkpoint_df.to_csv(

                CHECKPOINT_FILE,

                index=False

            )


            continue


print("\n" + "=" * 70)

print(
    "FULL EXPERIMENT PIPELINE FINISHED"
)

print("=" * 70)


print(

    "Completed experiments:",

    len(checkpoint_df)

)


print(

    "Checkpoint:",

    CHECKPOINT_FILE

)

STARTING FULL LDA EXPERIMENT PIPELINE
Total experiments: 495
Already completed: 474
Remaining: 21


PROCESSING FOLD 1
Fold 1, K=2: Already completed — skipping
Fold 1, K=3: Already completed — skipping
Fold 1, K=4: Already completed — skipping
Fold 1, K=5: Already completed — skipping
Fold 1, K=6: Already completed — skipping
Fold 1, K=7: Already completed — skipping
Fold 1, K=8: Already completed — skipping
Fold 1, K=9: Already completed — skipping
Fold 1, K=10: Already completed — skipping
Fold 1, K=11: Already completed — skipping
Fold 1, K=12: Already completed — skipping
Fold 1, K=13: Already completed — skipping
Fold 1, K=14: Already completed — skipping
Fold 1, K=15: Already completed — skipping
Fold 1, K=16: Already completed — skipping
Fold 1, K=17: Already completed — skipping
Fold 1, K=18: Already completed — skipping
Fold 1, K=19: Already completed — skipping
Fold 1, K=20: Already completed — skipping
Fold 1, K=21: Already completed — skipping
Fold 1, K=22: Already complete

In [ ]:

# ============================================================
# NG03-32 — Final Experiment Manifest
# ============================================================

final_manifest = checkpoint_df.copy()


final_manifest = final_manifest.sort_values(

    by=[
        "Fold",
        "Topics"
    ]

).reset_index(

    drop=True
)


FINAL_MANIFEST_FILE = (

    MODEL_DIR

    / "experiment_manifest_stage3.csv"

)


final_manifest.to_csv(

    FINAL_MANIFEST_FILE,

    index=False
)


print("=" * 70)

print(
    "FINAL STAGE-3 MANIFEST"
)

print("=" * 70)


print(
    "Shape:",
    final_manifest.shape
)


print()

display(
    final_manifest.head()
)


print()

print(
    f"✓ Final manifest saved:\n"
    f"{FINAL_MANIFEST_FILE}"
)

FINAL STAGE-3 MANIFEST
Shape: (495, 10)



,ExperimentID,Fold,Topics,TrainDocuments,TestDocuments,DictionarySize,ModelPath,TrainVectorPath,TestVectorPath,Status
0,NG_F1_K2,1,2,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
1,NG_F1_K3,1,3,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
2,NG_F1_K4,1,4,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
3,NG_F1_K5,1,5,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
4,NG_F1_K6,1,6,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed



✓ Final manifest saved:
/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/models/experiment_manifest_stage3.csv


In [ ]:

# ============================================================
# NG03-33 — Stage-3 Integrity Validation
# ============================================================

print("=" * 70)
print("NG03 STAGE-3 INTEGRITY VALIDATION")
print("=" * 70)


expected_experiments = (

    N_FOLDS

    * len(TOPIC_LIST)

)


actual_experiments = len(
    final_manifest
)


print(
    "Expected experiments:",
    expected_experiments
)


print(
    "Actual experiments:",
    actual_experiments
)


# ------------------------------------------------------------
# Check duplicates
# ------------------------------------------------------------

duplicate_count = (

    final_manifest.duplicated(

        subset=[
            "Fold",
            "Topics"
        ]

    ).sum()

)


print(
    "Duplicate Fold-K pairs:",
    duplicate_count
)


# ------------------------------------------------------------
# Check missing models
# ------------------------------------------------------------

missing_models = []


for _, row in final_manifest.iterrows():

    model_path = Path(
        row["ModelPath"]
    )


    if not model_path.exists():

        missing_models.append(

            row["ExperimentID"]

        )


print(
    "Missing models:",
    len(missing_models)
)


# ------------------------------------------------------------
# Check missing vectors
# ------------------------------------------------------------

missing_vectors = []


for _, row in final_manifest.iterrows():

    train_vector_path = Path(
        row["TrainVectorPath"]
    )


    test_vector_path = Path(
        row["TestVectorPath"]
    )


    if (

        not train_vector_path.exists()

        or

        not test_vector_path.exists()

    ):


        missing_vectors.append(

            row["ExperimentID"]

        )


print(
    "Missing vector files:",
    len(missing_vectors)
)


# ------------------------------------------------------------
# Overall result
# ------------------------------------------------------------

if (

    actual_experiments
    == expected_experiments

    and

    duplicate_count == 0

    and

    len(missing_models) == 0

    and

    len(missing_vectors) == 0

):


    print()

    print("=" * 70)

    print(
        "✓ NG03 COMPLETED SUCCESSFULLY"
    )

    print(
        "✓ ALL LDA MODELS VALID"
    )

    print(
        "✓ ALL TOPIC VECTORS VALID"
    )

    print(
        "✓ MANIFEST VALID"
    )

    print("=" * 70)


else:


    print()

    print(
        "⚠ NG03 VALIDATION FOUND ISSUES"
    )


    print(
        "Missing models:",
        missing_models[:10]
    )


    print(
        "Missing vectors:",
        missing_vectors[:10]
    )

NG03 STAGE-3 INTEGRITY VALIDATION
Expected experiments: 495
Actual experiments: 495
Duplicate Fold-K pairs: 0
Missing models: 0
Missing vector files: 0

✓ NG03 COMPLETED SUCCESSFULLY
✓ ALL LDA MODELS VALID
✓ ALL TOPIC VECTORS VALID
✓ MANIFEST VALID
